## 不使用@tool装饰器调用工具


In [2]:
## 定义一个函数作为工具
import math

def get_sqrt(num):
    return math.sqrt(num)


In [4]:
from langchain.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
import rich

model = ChatOpenAI(
    model="pdurugyan/qwen3.5-9b-deepseek-v4-flash-Q4_K_M-v_2:latest",
    api_key="None",
    base_url="http://localhost:11434/v1"
)

tmodel = model.bind_tools([get_sqrt])
msgs = [
    HumanMessage("9的平方根是多少")
]

resp = model.invoke(msgs)
tool_calls = resp.tool_calls
print(f"Tool Calls:{tool_calls}")
for tool_call in tool_calls:
    if tool_call["name"] == "get_sqrt":
        tresp = ToolMessage(
            content=get_sqrt(tool_call["args"].get('num')),
            tool_call_id=tool_call["id"],
            name=tool_call["name"]
        )

        msgs.append(tresp)
print("========================messages===============================")
for msg in msgs:
    msg.pretty_print()
print("========================messages===============================")
res = tmodel.invoke(msgs)
res.pretty_print()         

Tool Calls:[]
========================messages===============================
================================ Human Message =================================

9的平方根是多少
========================messages===============================
================================== Ai Message ==================================
Tool Calls:
  get_sqrt (call_0mvsk3fp)
 Call ID: call_0mvsk3fp
  Args:
    num: 9


### 下面就是docstring的使用，注意，Args的首字母必须大写，否则没有效果，Args上面的空行是必须的，否则也没有效果,冒号也必须是英文的，否则也没有效果

In [2]:
def check_weather(city:str):
    """ 
    查询天气信息

    Args:
        city : 具体的城市，如:北京等等

    Returns:
         返回城市的天气
          
    """

    return f"{city}天气晴朗，万里无云"

In [3]:
from langchain.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
import rich

model = ChatOpenAI(
    model="qwen3-vl:latest",
    api_key="None",
    base_url="http://localhost:11434/v1"
)

tmodel = model.bind_tools([check_weather])
msgs = [
    HumanMessage("北京的天气怎么样？")
]

resp = model.invoke(msgs)
tool_calls = resp.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "check_weather":
        tresp = ToolMessage(
            # content=check_weather(tool_call["args"].get('city')),
            content=check_weather(**tool_call["args"]),
            tool_call_id=tool_call["id"],
            name=tool_call["name"]
        )

        msgs.append(tresp)
print("========================messages===============================")
for msg in msgs:
    msg.pretty_print()
print("========================messages===============================")
res = tmodel.invoke(msgs)
res.pretty_print()    

========================messages===============================
================================ Human Message =================================

北京的天气怎么样？
========================messages===============================
================================== Ai Message ==================================
Tool Calls:
  check_weather (call_qywzy704)
 Call ID: call_qywzy704
  Args:
    city: 北京
